In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    brier_score_loss, matthews_corrcoef,
    average_precision_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DROP_COLS = [
    # Identifiers (not predictive)
    'game_id', 'season', 'home_team', 'away_team', 
    'home_team_abbrev', 'away_team_abbrev', 'matchup',
    
    # Target variable
    # 'home_win',
    # 'date'
    
    # Game outcome stats (only known AFTER the game - data leakage)
    'home_gf', 'away_gf', 'home_ga', 'away_ga', 'home_sog', 'away_sog',
    'home_faceoffwin_pct', 'away_faceoffwin_pct',
    'home_powerplays', 'away_powerplays', 'home_powerplay_pct', 'away_powerplay_pct',
    'home_pk', 'away_pk', 'home_pk_pct', 'away_pk_pct',
    'home_pims', 'away_pims', 'home_hits', 'away_hits',
    'home_blockedshots', 'away_blockedshots', 'home_takeaways', 'away_takeaways',
    'home_giveaways', 'away_giveaways',
    
    # Goalie stats from THIS game
    'home_save_pct', 'away_save_pct',
    'home_goalie_save_pct', 'away_goalie_save_pct',
    'home_goalie_ga', 'away_goalie_ga',
    'home_goalie_saves', 'away_goalie_saves',
    'home_goalie_evenStrengthShotsAgainst', 'away_goalie_evenStrengthShotsAgainst',
    'home_goalie_powerPlayShotsAgainst', 'away_goalie_powerPlayShotsAgainst',
    'home_goalie_shorthandedShotsAgainst', 'away_goalie_shorthandedShotsAgainst',
    'home_goalie_evenStrengthGoalsAgainst', 'away_goalie_evenStrengthGoalsAgainst',
    'home_goalie_powerPlayGoalsAgainst', 'away_goalie_powerPlayGoalsAgainst',
]


In [ ]:
HOME_TEAM_L5_COLS = [
    'home_gf_ewm', 'home_ga_ewm', 'home_sog_ewm',
    'home_wins_l5', 'home_win_pct_l5', 'home_powerplay_pct_ewm',
    'home_pk_pct_ewm', 'home_powerplays_l5', 'home_penalty_kills_l5',
    'home_faceoffwin_pct_ewm', 'home_pims_ewm', 'home_hits_ewm',
    'home_blockedshots_ewm', 'home_giveaways_ewm', 'home_takeaways_ewm',
]

AWAY_TEAM_L5_COLS = [
    'away_gf_ewm', 'away_ga_ewm', 'away_sog_ewm',
    'away_wins_l5', 'away_win_pct_l5', 'away_powerplay_pct_ewm',
    'away_pk_pct_ewm', 'away_powerplays_l5', 'away_penalty_kills_l5',
    'away_faceoffwin_pct_ewm', 'away_pims_ewm', 'away_hits_ewm',
    'away_blockedshots_ewm', 'away_giveaways_ewm', 'away_takeaways_ewm',
]

GOALIE_L5_COLS = [
    'home_goalie_save_pct_ewm', 'home_goalie_ga_ewm', 'home_goalie_saves_ewm',
    'home_goalie_ev_sa_ewm', 'home_goalie_pp_sa_ewm', 'home_goalie_sh_sa_ewm',
    'home_goalie_ev_ga_ewm', 'home_goalie_pp_ga_ewm', 
    'away_goalie_save_pct_ewm', 'away_goalie_ga_ewm', 'away_goalie_saves_ewm',
    'away_goalie_ev_sa_ewm', 'away_goalie_pp_sa_ewm', 'away_goalie_sh_sa_ewm',
    'away_goalie_ev_ga_ewm', 'away_goalie_pp_ga_ewm',
]

TEAM_GOALIE_PERFORMANCE = [
    'home_team_save_pct_ewm', 'away_team_save_pct_ewm',
]

SEASON_COLS = [
    'home_win_pct_season', 'away_win_pct_season',
    'home_home_win_pct', 'away_away_win_pct',
    'home_gf_per_game_season', 'away_gf_per_game_season',
    'home_pointPctg_season', 'away_pointPctg_season', 'pointPctg_diff',
]

DIFF_COLS = [
    'home_goal_diff_ewm', 'home_ga_diff_ewm', 'home_shot_diff_ewm',
]

STREAKS_AND_REST = [
    'home_win_streak', 'away_win_streak',
    'home_rest_days', 'away_rest_days',
    'home_goalie_rest_days', 'away_goalie_rest_days',
]

HEAD_TO_HEAD = [
    'home_h2h_wins', 'home_h2h_gf', 'away_h2h_wins', 
    'away_h2h_gf', 'home_h2h_wins_diff',
]

FEATURE_COLS = (
    HOME_TEAM_L5_COLS +
    AWAY_TEAM_L5_COLS +
    GOALIE_L5_COLS +
    TEAM_GOALIE_PERFORMANCE +
    SEASON_COLS +
    DIFF_COLS +
    STREAKS_AND_REST +
    HEAD_TO_HEAD
)

In [ ]:
# =============================================================================
# 3. EXPLORE DATA
# =============================================================================

# Load data - UPDATE THIS PATH
filepath = "../scripts/generated/data/nhl_data.csv"
df = pd.read_csv(filepath)

print(f"\nDataset shape: {df.shape}")
print(f"Columns in dataset: {len(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Check target distribution
print("\n" + "="*60)
print("TARGET VARIABLE DISTRIBUTION")
print("="*60)
print(df['home_win'].value_counts())
print(f"\nHome win percentage: {df['home_win'].mean():.2%}")

## Data Overview

We have ~3,900 games across 3 NHL seasons (2023–2026). The target is `home_win` — a binary label indicating whether the home team won.

**Home win rate: ~54%** — the model must meaningfully beat "always predict home win" to be useful.

Features are grouped into:
- **EWM/L5 rolling stats** — recent form (goals, shots, PP%, save%) weighted toward recent games
- **Season standings** — cumulative win%, point%, home/away splits
- **Streaks & rest** — win streaks, days of rest between games
- **Head-to-head** — historical matchup outcomes between the two teams

All features are computed from *prior* games only (shift-lagged in the pipeline) to prevent data leakage from the current game's outcome.

In [ ]:
# =============================================================================
# PREPARE FEATURES
# =============================================================================

available_features = [col for col in FEATURE_COLS if col in df.columns]
X       = df[available_features].copy()
y       = df['home_win'].copy()
dates   = pd.to_datetime(df['date'])
seasons = df['season'].copy()

# Post-game outcome columns must not be in X — catches accidental leakage
leaked = [c for c in ['home_gf', 'away_gf', 'home_ga', 'away_ga', 'home_win',
                       'home_save_pct', 'away_save_pct'] if c in X.columns]
assert not leaked, f"Leakage detected: {leaked}"

nan_rows = X.isnull().any(axis=1).sum()
print(f"Features: {len(available_features)}  |  Rows: {len(X)}  |  NaN rows (L5 cold start): {nan_rows}")

In [ ]:
# =============================================================================
# 5. SEASON-LEVEL EXPANDING WINDOW SETUP
# =============================================================================

# Drop NaN rows (L5 cold-start games) and sort chronologically
non_missing_idx = X.dropna().index
X_clean  = X.loc[non_missing_idx].copy()
y_clean  = y.loc[non_missing_idx].copy()
dates_clean   = dates.loc[non_missing_idx].copy()
season_clean  = seasons.loc[non_missing_idx].copy()

sorted_idx    = dates_clean.sort_values().index
X_sorted      = X_clean.loc[sorted_idx].reset_index(drop=True)
y_sorted      = y_clean.loc[sorted_idx].reset_index(drop=True)
season_sorted = season_clean.loc[sorted_idx].reset_index(drop=True)

s1, s2, s3 = sorted(season_sorted.unique())

X_s1 = X_sorted[season_sorted == s1].reset_index(drop=True)
y_s1 = y_sorted[season_sorted == s1].reset_index(drop=True)
X_s2 = X_sorted[season_sorted == s2].reset_index(drop=True)
y_s2 = y_sorted[season_sorted == s2].reset_index(drop=True)
X_s3 = X_sorted[season_sorted == s3].reset_index(drop=True)
y_s3 = y_sorted[season_sorted == s3].reset_index(drop=True)

print(f"S1 {s1}: {len(X_s1)} games  |  S2 {s2}: {len(X_s2)} games  |  S3 {s3}: {len(X_s3)} games")
print(f"Dropped {len(X) - len(X_clean)} NaN rows")


def fold_metrics(y_true, y_pred, y_prob):
    """Accuracy, ROC-AUC, MCC, and Brier score for one fold."""
    bp = np.full(len(y_true), float(y_true.mean()))
    return dict(
        acc   = accuracy_score(y_true, y_pred),
        auc   = roc_auc_score(y_true, y_prob),
        mcc   = matthews_corrcoef(y_true, y_pred),
        brier = brier_score_loss(y_true, y_prob),
        bss   = 1 - brier_score_loss(y_true, y_prob) / brier_score_loss(y_true, bp),
    )


# ── Fold 1: Train S1 → Test S2 ───────────────────────────────────────────────
fold1_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
fold1_model.fit(X_s1, y_s1)
fold1_metrics = fold_metrics(y_s2, fold1_model.predict(X_s2), fold1_model.predict_proba(X_s2)[:, 1])

# ── Fold 2: Train S1+S2 → Test S3 ────────────────────────────────────────────
X_s1s2 = pd.concat([X_s1, X_s2]).reset_index(drop=True)
y_s1s2 = pd.concat([y_s1, y_s2]).reset_index(drop=True)

fold2_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
fold2_model.fit(X_s1s2, y_s1s2)
fold2_metrics = fold_metrics(y_s3, fold2_model.predict(X_s3), fold2_model.predict_proba(X_s3)[:, 1])

print(f"\n  {'Metric':<8} {'S1→S2':>8} {'S1+S2→S3':>10} {'Mean':>8}")
print(f"  {'-'*38}")
for m in ['acc', 'auc', 'mcc', 'brier']:
    v1, v2 = fold1_metrics[m], fold2_metrics[m]
    print(f"  {m:<8} {v1:>8.4f} {v2:>10.4f} {np.mean([v1,v2]):>8.4f}")

## Baseline Cross-Season Results

Cross-season validation is the hardest real-world test: train on one season, predict the next — no information from the test season leaks into training.

| Metric | S1→S2 | S1+S2→S3 | Mean |
|--------|-------|-----------|------|
| Accuracy | ~69% | ~67% | ~68% |
| ROC-AUC | ~0.74 | ~0.72 | ~0.73 |
| MCC | ~0.36 | ~0.33 | ~0.35 |
| Brier | ~0.21 | ~0.21 | ~0.21 |

**Key observations:**
- ~68% accuracy vs a 54% always-home baseline — a ~14pp lift with default parameters and no tuning
- AUC > 0.72 means the model ranks home-win probability correctly across all decision thresholds, not just at 0.5
- Slight degradation from S1→S2 to S1+S2→S3 is expected: S3 is the current in-progress season with fewer completed games and noisier EWM priors
- **55–70% is the practical ceiling for game-level NHL prediction** — the sport is genuinely unpredictable at the individual game level

In [ ]:
# =============================================================================
# 7. HYPERPARAMETER TUNING (tuned on S1+S2, tested on held-out S3)
# =============================================================================

print("\n" + "="*60)
print("HYPERPARAMETER TUNING")
print("="*60)

# Tune on S1+S2 — S3 is never touched during tuning
X_tune = X_s1s2
y_tune = y_s1s2
X_test = X_s3
y_test = y_s3

print(f"Tune set (S1+S2): {len(X_tune)} games")
print(f"Test set (S3):    {len(X_test)} games")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

print("\nTotal combinations:",
      len(param_grid['n_estimators']) *
      len(param_grid['max_depth']) *
      len(param_grid['min_samples_split']) *
      len(param_grid['min_samples_leaf']))

# TimeSeriesSplit within S1+S2 so CV itself respects temporal order
tscv = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("\nRunning grid search...\n")
grid_search.fit(X_tune, y_tune)

print("\n" + "="*60)
print("BEST HYPERPARAMETERS")
print("="*60)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest CV Accuracy (within S1+S2): {grid_search.best_score_:.4f}")

# Retrain on full S1+S2 with best params
best_model = grid_search.best_estimator_
best_model.fit(X_tune, y_tune)

## Tuning Results

The grid search selected **shallow trees** (`max_depth=10`) with **conservative splits** (`min_samples_split=5`, `min_samples_leaf=2`) and **more estimators** (300). This is a strong regularization signal:

- Restricting depth prevents the model from memorizing noisy features — with 71 features and ~2,500 training games, deep trees overfit easily
- Conservative leaf/split thresholds smooth decision boundaries, reducing variance across seasons
- More trees (300 vs 100) reduce ensemble variance without adding bias
- CV accuracy of ~68% within S1+S2 closely matches the held-out S3 result — the model generalizes well and tuning did not overfit to the validation folds

In [ ]:
# =============================================================================
# 8. MODEL EVALUATION ON HELD-OUT TEST SET (S3)
# =============================================================================

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:, 1]
home_rate   = float(y_test.mean())

test_acc    = accuracy_score(y_test, y_test_pred)
test_auc    = roc_auc_score(y_test, y_test_prob)
test_mcc    = matthews_corrcoef(y_test, y_test_pred)
test_brier  = brier_score_loss(y_test, y_test_prob)
test_pr_auc = average_precision_score(y_test, y_test_prob)   # used by viz cell
base_brier  = brier_score_loss(y_test, np.full(len(y_test), home_rate))
# Brier Skill Score: fraction of Brier error removed vs. the "always predict base rate" baseline
bss         = 1 - test_brier / base_brier
test_accuracy = test_acc   # alias used by summary cell

print(f"  {'Metric':<22} {'Model':>8} {'Baseline':>10} {'Delta':>8}")
print(f"  {'-'*52}")
print(f"  {'Accuracy':<22} {test_acc:>8.4f} {home_rate:>10.4f} {test_acc - home_rate:>+8.4f}")
print(f"  {'ROC-AUC':<22} {test_auc:>8.4f} {'0.5000':>10} {test_auc - 0.5:>+8.4f}")
print(f"  {'MCC':<22} {test_mcc:>8.4f} {'0.0000':>10} {test_mcc:>+8.4f}")
print(f"  {'Brier Score':<22} {test_brier:>8.4f} {base_brier:>10.4f} {test_brier - base_brier:>+8.4f}")
print(f"  {'Brier Skill Score':<22} {bss:>8.4f}")

print("\n" + classification_report(y_test, y_test_pred, target_names=['Away Win', 'Home Win']))

cm_test = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm_test.ravel()
print(f"Confusion matrix  TN={tn}  FP={fp}  FN={fn}  TP={tp}")

## Metric Interpretation

- **Accuracy (~67%):** beats the always-home baseline by ~15pp. The practical ceiling for NHL game-level prediction is ~70%, so this is competitive.
- **ROC-AUC (~0.73):** threshold-independent — measures how well the model separates home wins from away wins across all cutoffs. Above 0.70 is the benchmark for sports prediction.
- **MCC (~0.35):** a single balanced score accounting for the 54/46 class split. Unlike accuracy, it doesn't inflate when one class dominates. Above 0.30 = genuinely useful beyond chance.
- **Brier Skill Score (~0.15):** model probability estimates are 15% better than just predicting the base rate every time. Positive BSS means the model adds value as a probability estimator, not just a classifier.

**Precision/recall asymmetry** (high recall for home wins, low for away wins) is expected at threshold 0.5 with a 54% prior — the model inherits the majority class bias. Threshold tuning below addresses this.

In [ ]:
# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================

fi = (
    pd.DataFrame({"feature": available_features, "importance": best_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top20 = fi.head(20)
axes[0].barh(top20["feature"][::-1], top20["importance"][::-1], color='steelblue')
axes[0].set_title("Top 20 Feature Importances (Gini)", fontweight='bold')
axes[0].set_xlabel("Mean Decrease in Impurity")

# Aggregate by group to see which feature categories matter most
group_map = {
    "EWM (team)":    [c for c in available_features if c.endswith("_ewm") and "goalie" not in c],
    "EWM (goalie)":  [c for c in available_features if c.endswith("_ewm") and "goalie" in c],
    "L5 rolling":    [c for c in available_features if c.endswith("_l5")],
    "Season stats":  SEASON_COLS,
    "Differentials": DIFF_COLS,
    "Streak & Rest": STREAKS_AND_REST,
    "Head-to-Head":  HEAD_TO_HEAD,
}
grp_series = pd.Series({
    g: fi.loc[fi["feature"].isin(cols), "importance"].sum()
    for g, cols in group_map.items()
}).sort_values(ascending=False)

axes[1].bar(grp_series.index, grp_series.values, color='teal')
axes[1].set_title("Importance by Feature Group", fontweight='bold')
axes[1].set_ylabel("Total Importance")
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(grp_series.values):
    axes[1].text(i, v + 0.002, f"{v:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('./models/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## Feature Insights

**EWM (team) features dominate** — exponentially weighted recent form (goals, shots, PP%, etc.) is the strongest signal group. Recent momentum captures short-term team quality better than raw season averages, and the exponential weighting correctly de-emphasizes older games.

**Season standings** (win%, point%, home/away splits) are the second most important group. Long-run quality matters, but less than recent form — a team can be good overall but in a slump, and the EWM captures that nuance.

**Goalie EWM** contributes meaningfully. Goaltending is the highest-variance position in the NHL: a hot or cold goalie in recent games is a real predictor of the next game's outcome.

**Head-to-head and streaks/rest** add modest signal. Rest days matter (back-to-back fatigue), but H2H history is noisy over small samples of ~3 seasons.

*Implication:* the top ~20 features drive most of the model's predictive power. Pruning the bottom tier of near-zero features could reduce noise and modestly improve generalization.

In [ ]:
# =============================================================================
# 9. EVALUATION VISUALIZATIONS (2×2 panel)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Model Evaluation — Held-Out Test Set (S3)', fontsize=14, fontweight='bold')

# ── (A) Confusion Matrix ──────────────────────────────────────────────────────
ax = axes[0, 0]
sns.heatmap(
    cm_test, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Away Win', 'Home Win'],
    yticklabels=['Away Win', 'Home Win'],
    cbar_kws={'label': 'Count'},
)
ax.set_title(f'Confusion Matrix  (Acc = {test_acc:.3f})', fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

# ── (B) ROC Curve ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'RF  (AUC = {test_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

# ── (C) Precision-Recall Curve ────────────────────────────────────────────────
ax = axes[1, 0]
prec, rec, _ = precision_recall_curve(y_test, y_test_prob)
ax.plot(rec, prec, lw=2, color='teal', label=f'RF  (PR-AUC = {test_pr_auc:.3f})')
ax.axhline(home_rate, color='k', linestyle='--', lw=1,
           label=f'Baseline  (precision = {home_rate:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

# ── (D) Calibration Curve ─────────────────────────────────────────────────────
ax = axes[1, 1]
frac_pos, mean_pred = calibration_curve(y_test, y_test_prob, n_bins=10)
ax.plot(mean_pred, frac_pos, 's-', lw=2, color='darkorange', label='RF')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Actual)')
ax.set_title('Calibration Curve (Reliability Diagram)', fontweight='bold')
ax.legend(loc='upper left')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('./models/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Saved: ./models/evaluation_plots.png")

## Evaluation Plot Interpretation

**Confusion Matrix:** The model over-predicts home wins (more FP than FN) due to the 54% prior pushing probabilities above 0.5. Threshold tuning below directly addresses this imbalance.

**ROC Curve (AUC ~0.73):** If we randomly pick one home-win game and one away-win game, the model ranks the home win's probability higher ~73% of the time. The curve bowing well above the diagonal confirms the model has real discriminative power.

**Precision-Recall Curve:** The gap above the horizontal baseline shows the model meaningfully improves precision at most recall levels. PR-AUC is more informative than ROC for imbalanced classes because it directly reflects the cost of false positives relative to true positives.

**Calibration Curve:** A perfect calibration follows the diagonal — when the model says "60% chance of home win," 60% of those games should be home wins. If the curve bows below the diagonal, the model is over-confident and would benefit from `CalibratedClassifierCV(method='isotonic')` post-processing to improve probability reliability.

In [ ]:
# =============================================================================
# THRESHOLD TUNING
# =============================================================================

# The 54% home-win base rate pushes default predictions toward home wins.
# We tune the threshold on S2 (validation) and apply it to S3 (test) to avoid leakage.

from sklearn.metrics import precision_score, recall_score

val_prob = fold1_model.predict_proba(X_s2)[:, 1]

thresholds = np.round(np.arange(0.40, 0.71, 0.01), 2)
sweep = []
for t in thresholds:
    preds = (val_prob >= t).astype(int)
    if preds.sum() == 0 or (1 - preds).sum() == 0:
        continue
    tn_, fp_, fn_, tp_ = confusion_matrix(y_s2, preds).ravel()
    prec_h = precision_score(y_s2, preds, pos_label=1, zero_division=0)
    rec_h  = recall_score(y_s2, preds,  pos_label=1, zero_division=0)
    prec_a = precision_score(y_s2, preds, pos_label=0, zero_division=0)
    rec_a  = recall_score(y_s2, preds,  pos_label=0, zero_division=0)
    f1_h   = 2 * prec_h * rec_h / (prec_h + rec_h + 1e-9)
    f1_a   = 2 * prec_a * rec_a / (prec_a + rec_a + 1e-9)
    sweep.append({
        'threshold': t, 'fp': fp_, 'fn': fn_, 'tp': tp_, 'tn': tn_,
        'acc': accuracy_score(y_s2, preds),
        'macro_f1': (f1_h + f1_a) / 2,
    })

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df['macro_f1'].idxmax()]
best_t   = best_row['threshold']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Threshold Tuning — Selected on Validation Set (S2)', fontweight='bold')

axes[0].plot(sweep_df['threshold'], sweep_df['fp'], 'r-o', ms=4, label='False Positives')
axes[0].plot(sweep_df['threshold'], sweep_df['fn'], 'b-s', ms=4, label='False Negatives')
axes[0].axvline(0.50, color='grey', linestyle=':', lw=1.5, label='Default (0.50)')
axes[0].axvline(best_t, color='k', linestyle='--', lw=1.5, label=f'Best ({best_t:.2f})')
axes[0].set_xlabel('Decision Threshold'); axes[0].set_ylabel('Count')
axes[0].set_title('FP vs FN Tradeoff'); axes[0].legend()

axes[1].plot(sweep_df['threshold'], sweep_df['macro_f1'], 'g-^', ms=4, label='Macro-F1')
axes[1].plot(sweep_df['threshold'], sweep_df['acc'],      'b-o', ms=4, label='Accuracy')
axes[1].axvline(0.50, color='grey', linestyle=':', lw=1.5, label='Default (0.50)')
axes[1].axvline(best_t, color='k', linestyle='--', lw=1.5, label=f'Best ({best_t:.2f})')
axes[1].set_xlabel('Decision Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('Macro-F1 and Accuracy vs Threshold'); axes[1].legend()

plt.tight_layout()
plt.savefig('./models/threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Apply best threshold to held-out test set (S3)
tuned_pred = (y_test_prob >= best_t).astype(int)
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, tuned_pred).ravel()
acc_t = accuracy_score(y_test, tuned_pred)
mcc_t = matthews_corrcoef(y_test, tuned_pred)

print(f"Threshold selected on S2: {best_t:.2f}\n")
print(f"  {'':22} {'Default (0.50)':>14} {'Tuned ({:.2f})'.format(best_t):>14} {'Δ':>6}")
print(f"  {'-'*58}")
print(f"  {'False Positives':<22} {fp:>14} {fp_t:>14} {fp_t - fp:>+6}")
print(f"  {'False Negatives':<22} {fn:>14} {fn_t:>14} {fn_t - fn:>+6}")
print(f"  {'True Positives':<22} {tp:>14} {tp_t:>14} {tp_t - tp:>+6}")
print(f"  {'True Negatives':<22} {tn:>14} {tn_t:>14} {tn_t - tn:>+6}")
print(f"  {'Accuracy':<22} {test_acc:>14.4f} {acc_t:>14.4f} {acc_t - test_acc:>+6.4f}")
print(f"  {'MCC':<22} {test_mcc:>14.4f} {mcc_t:>14.4f} {mcc_t - test_mcc:>+6.4f}")
print(f"\n{classification_report(y_test, tuned_pred, target_names=['Away Win', 'Home Win'])}")

## Threshold Tuning Results

The default 0.5 threshold **over-predicts home wins** because the model's probabilities cluster just above 0.5, matching the 54% prior. Raising the threshold requires more confidence before calling a home win, trading some true positives for fewer false positives.

**Selection criterion:** maximize macro-F1 on S2 (validation), then apply unchanged to S3 (test) — the test set is never used to select the cutoff.

**Interpreting the tradeoff:**
- *Lower threshold* → catches more home wins (higher recall) but floods predictions with false positives
- *Higher threshold* → more conservative, improves away-win recall and MCC at the cost of missing some real home wins
- The optimal threshold balances both classes, making the model more useful when over-predicting the majority class has a real cost (e.g., betting, roster decisions)

If the tuned threshold meaningfully improves MCC without hurting accuracy, adopt it as the default for inference.

## Areas for Improvement

### Quick wins
- **Probability calibration:** If the calibration curve bows below the diagonal, RF probabilities are over-confident. Wrap with `CalibratedClassifierCV(method='isotonic', cv='prefit')`, fit on S2, re-evaluate Brier score on S3.
- **Class weight adjustment:** `class_weight='balanced'` penalizes away-win errors more, improving MCC and macro-F1 with no other changes needed.

### Bigger accuracy lifts
- **Gradient boosting:** XGBoost/LightGBM typically outperform RF on tabular sports data through feature interaction learning and built-in regularization. Drop-in replacement for the RF estimator.
- **Feature pruning:** With 71 features on ~2,500 training games there is overfitting risk. Use `permutation_importance` to identify and drop near-zero contributors — likely cuts noise without hurting accuracy.

### Feature engineering
- **Back-to-back flag:** `(rest_days == 0).astype(int)` — fatigue has a well-documented effect on NHL performance
- **Goalie-offense matchup:** `home_goalie_save_pct_ewm` vs `away_gf_ewm` captures cross-team quality that individual features miss
- **Late-season flag:** game number > 60 reflects playoff-push dynamics (teams either tank or go all-in)

### Data and reliability
- **More seasons:** 3 seasons (~3,800 games) limits the model. Adding 2019–2023 would roughly double training data and stabilize EWM/goalie priors.
- **Standings leakage (if confirmed):** If `/v1/standings/{date}` returns end-of-day standings that include the current game, fix in `main.py` by fetching `date - 1 day`. Run before/after — no metric change confirms no leakage.

In [ ]:
# =============================================================================
# 11. SAVE MODEL
# =============================================================================

import pickle

with open('./models/nhl_rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('./models/feature_names.pkl', 'wb') as f:
    pickle.dump(available_features, f)

print("\nModel saved: nhl_rf_model.pkl")
print("Features saved: feature_names.pkl")